In [11]:
from pathlib import Path
import math
import pandas as pd
from pyproj import Geod

## Load data and eda

In [8]:
# load data from https://craters.sjrdesign.net/
df = pd.read_csv("/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv")
print(df.shape)
df.head()

(385049, 34)


/var/folders/bh/1rv8574s3nl7smf72s65w90m0000gn/T/ipykernel_16504/2841572640.py:2: DtypeWarning: Columns (21,22,23,24,25,26,27,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv")


,CRATER_ID,LAT_CIRC_IMG,LON_CIRC_IMG,LAT_ELLI_IMG,LON_ELLI_IMG,DIAM_CIRC_IMG,DIAM_CIRC_SD_IMG,DIAM_ELLI_MAJOR_IMG,DIAM_ELLI_MINOR_IMG,DIAM_ELLI_ECCEN_IMG,...,LAY_MORPH3,LAY_NOTES,INT_MORPH1,INT_MORPH2,INT_MORPH3,CONF,NOTES,DEG_RIM,DEG_EJC,DEG_FLR
0,01-1-000003,79.257599,211.909949,79.257584,211.910021,19.526698,0.230187,19.746718,19.286041,0.214742,...,NaN,NaN,CpxUnc,NaN,Floor Deposits,NaN,NaN,1.0,1.0,1.0
1,01-1-000004,78.696512,207.162344,78.696517,207.162105,4.232291,0.089167,4.449180,4.019798,0.428605,...,NaN,NaN,NaN,NaN,Floor Deposits,NaN,NaN,3.0,2.0,1.0
2,01-1-000012,77.791422,186.586023,77.791443,186.585690,8.542042,0.193030,8.818632,8.251528,0.352817,...,NaN,NaN,CpxFF,NaN,NaN,NaN,NaN,2.0,2.0,2.0
3,01-1-000013,76.655040,194.807812,76.655020,194.807973,7.967843,0.139290,8.268589,7.662693,0.375745,...,NaN,NaN,CpxFF,NaN,NaN,NaN,NaN,3.0,2.0,3.0
4,01-1-000014,76.976285,195.615795,76.976294,195.616254,20.948878,0.340399,21.550893,20.326902,0.332212,...,NaN,NaN,CpxFF,NaN,Floor Deposits,NaN,NaN,3.0,2.0,3.0


## generating 5ºx5º tiles

In [16]:
CSV_FILE      = "/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv"
OUT_DIR       = Path("tiles_scc")                         # where .scc files go
MIN_DIAM_KM   = 1.0                                       # optional cut-off

# IAU 2000 Mars ellipsoid (km)  –– you can change if you prefer 2009 values
A_EQUAT = 3396.19     # semi-major axis
B_POLAR = 3376.20     # semi-minor axis
GEOD    = Geod(a=A_EQUAT * 1000, b=B_POLAR * 1000)        # metres  ➜ pyproj

# ----------------------------------------------------------------------
# 2.  Read & lightly clean the input catalogue
# ----------------------------------------------------------------------
print("Loading crater catalogue …")
df = pd.read_csv(CSV_FILE, engine="python")

defaults = {
    "DEPTH_RIMFLOOR_TOPOG": 0.0,
    "DEG_RIM":              1,
    "DEG_EJC":              1,
    "DEG_FLR":              1,
}

for col, default in defaults.items():
    if col not in df.columns:             # column absent ➜ create it
        df[col] = default
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(default)

# ----------------------------------------------------------------------
# 3.  Assign every crater to a 5°×5° tile  (lower-left corner as key)
# ----------------------------------------------------------------------
df["tile_lat"] = (df["LAT_CIRC_IMG"] // 5 * 5).astype(int)
df["tile_lon"] = (df["LON_CIRC_IMG"] // 5 * 5).astype(int)   # 0–360 °E works too

# ----------------------------------------------------------------------
# 4.  Helper: exact ellipsoidal area of a tile
# ----------------------------------------------------------------------
def tile_area_km2(lat0: int, lon0: int) -> float:
    """
    Four corners   (lat0,lon0)  … (lat0,lon0+5) … (lat0+5,lon0+5) … (lat0+5,lon0)
    Returns area in square-kilometres (always positive)
    """
    lats = [lat0,        lat0,        lat0 + 5,  lat0 + 5]
    lons = [lon0,        lon0 + 5,    lon0 + 5,  lon0]
    area_m2, _ = GEOD.polygon_area_perimeter(lons, lats)
    return abs(area_m2) / 1_000_000   # m² ➜ km²

# ----------------------------------------------------------------------
# 5.  Write every tile’s structured .scc file
# ----------------------------------------------------------------------
OUT_DIR.mkdir(exist_ok=True)

for (lat0, lon0), tile in df.groupby(["tile_lat", "tile_lon"]):
    area = tile_area_km2(lat0, lon0)
    fname = OUT_DIR / f"tile_lat{lat0:+03d}_lon{lon0:+04d}.scc"

    # build header exactly like Pickering.scc
    header = [
        "# craterstats 3",
        "# structured crater list",
        f"# area={area:.6f}",
        "# columns: diameter latitude longitude depth rim_conf ejecta_conf floor_conf"
    ]

    # choose the seven columns (order matters!)
    cols = ["DIAM_CIRC_IMG", "LAT_CIRC_IMG", "LON_CIRC_IMG",
            "DEPTH_RIMFLOOR_TOPOG", "DEG_RIM", "DEG_EJC", "DEG_FLR"]

    rows = tile[cols].round(6).astype(str).agg(" ".join, axis=1).tolist()

    fname.write_text("\n".join(header + rows), encoding="utf-8")
    print(f"✓ {fname.name:35s}  ({len(tile):4d} craters,  area {area:8.1f} km²)")

print("\nAll tiles written to", OUT_DIR.resolve())


Loading crater catalogue …
✓ tile_lat-90_lon+000.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+005.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+045.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+095.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+145.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+150.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+225.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+235.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+245.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+250.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+255.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+270.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+345.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-

In [17]:
# ------------------------------------------------------------------
# 0.  Imports & parameters
# ------------------------------------------------------------------
import pandas as pd
from pathlib import Path
from datetime import date
from pyproj import Geod

CSV_FILE    = "/Users/jessica/Documents/EPSS/MARS_CRATER_DENSITY/Catalog_Mars_Release_2020_1kmPlus_FullMorphData.csv"
OUT_DIR     = Path("tiles_scc")                # where *.scc files go
MIN_DIAM_KM = 1.0                              # optional diameter cutoff

# IAU-2000 Mars reference ellipsoid (km)
A_EQUAT = 3396.19
B_POLAR = 3376.20
GEOD    = Geod(a=A_EQUAT*1000, b=B_POLAR*1000)   # metres  → pyproj

# ------------------------------------------------------------------
# 1.  Load & quick-clean the catalogue
# ------------------------------------------------------------------
print("Loading crater catalogue …")
df = pd.read_csv(CSV_FILE, engine="python")

# force the three essential numeric columns
num_cols = ["DIAM_CIRC_IMG", "LAT_CIRC_IMG", "LON_CIRC_IMG"]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

# optional ≥ 1 km filter
if MIN_DIAM_KM:
    df = df[df["DIAM_CIRC_IMG"] >= MIN_DIAM_KM]

# assign each crater to a 5 °×5 ° tile (lower-left corner)
df["tile_lat"] = (df["LAT_CIRC_IMG"] // 5 * 5).astype(int)
df["tile_lon"] = (df["LON_CIRC_IMG"] // 5 * 5).astype(int)

# ------------------------------------------------------------------
# 2.  Helpers ─ boundary, area
# ------------------------------------------------------------------
def tile_boundary(lat0: int, lon0: int):
    """Return the five (lon,lat) vertices that close the square tile."""
    return [
        (lon0,     lat0),
        (lon0+5,   lat0),
        (lon0+5,   lat0+5),
        (lon0,     lat0+5),
        (lon0,     lat0)      # closes polygon
    ]

def tile_area_km2(lat0: int, lon0: int) -> float:
    """Exact ellipsoidal area of the 5°×5° tile in km²."""
    lons, lats = zip(*tile_boundary(lat0, lon0)[:-1])   # four corners only
    area_m2, _ = GEOD.polygon_area_perimeter(lons, lats)
    return abs(area_m2) / 1_000_000

# ------------------------------------------------------------------
# 3.  Write every tile as a Pickering-style *.scc file
# ------------------------------------------------------------------
OUT_DIR.mkdir(exist_ok=True)
today = date.today().strftime("%d.%m.%Y")

for (lat0, lon0), tile in df.groupby(["tile_lat", "tile_lon"]):
    area = tile_area_km2(lat0, lon0)
    fname = OUT_DIR / f"tile_lat{lat0:+03d}_lon{lon0:+04d}.scc"

    # ---- header ---------------------------------------------------
    header = [
        "# Spatial crater count",
        "#",
        f"# Date of measurement = {today}",
        "#",
        "# Ellipsoid axes:",
        f"a_axis_radius = {A_EQUAT:.2f} <km>",
        f"b_axis_radius = {A_EQUAT:.2f} <km>",
        f"c_axis_radius = {A_EQUAT:.2f} <km>",
        "coordinate_system_name = unknown",
        "#",
        "# area_shapes:",
        "unit_boundary = {vertex,sub_area,tag,lon,lat",
        "#",
        f"# Area_name1 = tile {lat0}/{lon0}",
        f"# Area_size1 = {area:.11f} <km^2>",
        "#",
    ]

    # ---- polygon vertices ----------------------------------------
    bnd = [
        f"{i+1}\t1\text\t{lon:.12f}\t{lat:.12f}"
        for i, (lon, lat) in enumerate(tile_boundary(lat0, lon0))
    ] + ["}"]          # closes unit_boundary

    # ---- crater diameters section --------------------------------
    crater_block = [
        "#",
        f"Total_area = {area:.11f} <km^2>",
        "#",
        "# crater_diameters:",
        "crater = {diam,lon,lat"
    ]

    crater_rows = (
        tile[["DIAM_CIRC_IMG", "LON_CIRC_IMG", "LAT_CIRC_IMG"]]
        .round(6)
        .astype(str)
        .agg("\t".join, axis=1)
        .tolist()
        + ["}"]        # closes crater list
    )

    # ---- write file ----------------------------------------------
    fname.write_text(
        "\n".join(header + bnd + crater_block + crater_rows),
        encoding="utf-8"
    )

    print(f"✓ {fname.name:35s}  ({len(tile):4d} craters,  area {area:8.1f} km²)")

print("\nAll tiles written to", OUT_DIR.resolve())


Loading crater catalogue …
✓ tile_lat-90_lon+000.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+005.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+045.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+095.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+145.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+150.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+225.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+235.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+245.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+250.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+255.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+270.scc              (   1 craters,  area   3870.4 km²)
✓ tile_lat-90_lon+345.scc              (   2 craters,  area   3870.4 km²)
✓ tile_lat-

## References

https://craters.sjrdesign.net/

https://home.csulb.edu/~rodrigue/geog441541/lectures/midterm/marsinspace.html#:~:text=Equatorial%20radius:%203%2C396%20km%20(Earth,km3%20(Earth:%201%2C083%2C200%2C000%2C000%20km

https://en.wikipedia.org/wiki/Spherical_cap

https://arxiv.org/pdf/1109.4448